# CODEX cell type detection → per-type scRNA-seq mapping

Sets cell types from markers in a cell-level CODEX table (csv: marker intensities + x, y), then maps scRNA-seq cell expression onto CODEX cell positions **within the same type**.

| Step | Module | Content |
|---|---|---|
| 1 | `codex_table` | csv → AnnData (markers = X, coordinates = obsm['spatial'], annotations = obs) |
| 2–6 | `codex_types` | preprocessing → clustering (Leiden) → cluster annotation by marker scores → re-splitting mixed clusters → checks |
| 7 | | save (h5ad, csv) |
| 8 | `rna_prep` | scRNA raw → cell calling → QC → doublet removal → normalisation → HVG → Harmony integration → clusters → types (RNA only) |
| 9 | `rna_map` | linked features → mapping plan → kNN / scMODAL mapping → held-out marker validation |
| 10 | `maxfuse_map` | MaxFuse cell matching → transfer scRNA labels to CODEX → tissue position per scRNA cell |
| 11 | `niche_remap` | type-mean mapping → within-type remapping by NicheNet (similarity as fallback) hypotheses → judged with CODEX signal-response proteins |
| 12 | `map_benchmark` | benchmark: comparing mapping methods by held-out marker tests (same folds·shared cells) · niche remapping judgement summary (passes vs chance, gain over random placement) |

Drive layout: `WORK_DIR/modules` (py), `inputs` (csv, scRNA h5ad), `outputs`, `resources` (cache)

## 0. Settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/CODEX_RNA'
import os, sys
for d in ('modules', 'inputs', 'outputs', 'resources'):
    os.makedirs(f'{WORK_DIR}/{d}', exist_ok=True)
sys.path.insert(0, f'{WORK_DIR}/modules')
OUT = f'{WORK_DIR}/outputs'

In [ ]:
# packages needed for clustering·integration (works without scanpy — avoids Colab's scanpy/scipy version conflict)
# pin the numpy·pandas·scipy versions installed in Colab. Without pinning, recent anndata (requires pandas ≥ 2.3)
# upgrades pandas and conflicts with google-colab → with pins, pip picks a matching anndata (0.12.x)
import os, tempfile, numpy, pandas, scipy
PINS = os.path.join(tempfile.gettempdir(), 'colab_pins.txt')
with open(PINS, 'w') as f:
    f.write(f'numpy=={numpy.__version__}\npandas=={pandas.__version__}\nscipy=={scipy.__version__}\n')
print(open(PINS).read())
!pip -q install -c {PINS} anndata igraph leidenalg pynndescent harmonypy umap-learn maxfuse pyreadr

In [ ]:
import importlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
import codex_table as ct, codex_types as cty, rna_prep as rp, rna_map as rm, maxfuse_map as mfm, niche_remap as nr
for m in (ct, cty, rp, rm, mfm, nr):
    importlib.reload(m)
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 200)

## 1. Loading the CODEX csv

- First check column roles (markers / coordinates / annotations) and sample names with `describe`.
- STELLAR tonsil·Barrett's oesophagus table: 44 markers (already z-scored), `x, y` (pixels), `sample_name` (tonsil / Barretts Esophagus), `cell_type` (given labels — used only for comparison).
- `UM_PER_PX`: factor converting coordinates to µm. Enter after checking the dataset README (commonly 0.377 µm/px on CODEX instruments). If unknown, None (pixels as is).

In [ ]:
CSV = f'{WORK_DIR}/inputs/BE_Tonsil_l3_dryad.csv'
SAMPLE = {'sample_name': 'tonsil'}     # one tissue only (None = all)
UM_PER_PX = None                      # e.g. 0.377

df = ct.read_codex_csv(CSV, cache_dir=f'{WORK_DIR}/resources')
roles = ct.describe(df)

In [ ]:
adata = ct.to_anndata(df, subset=SAMPLE, um_per_px=UM_PER_PX)
del df
adata

## 2. Preprocessing · exclusive-marker check

- Tables already z-scored are clipped only from above at 5 (so a few extreme values do not drag clusters). Raw intensities → arcsinh → z-score (detected automatically).
- `spill`: cells positive for both markers of a pair that cannot co-occur in one cell (CD3+CD20 etc.). Flags neighbour spillover·segmentation errors — not removed, only left out of mapping.
  Normal range if `both` is below `expected` (fraction overlapping by chance).

In [ ]:
cty.preprocess(adata)
spill_tab = cty.spillover_flags(adata)
spill_tab

## 3. Clustering

- PCA (20) → kNN (15) → Leiden. About 1–3 min for 170 k cells on 2 CPUs.
- Larger `RESOLUTION` splits more finely. Annotation uses cluster means, so a somewhat fine split (1.5–3) preserves rare types.
- `CLUSTER_MARKERS`: markers used for clustering (None = all). Better to leave out markers unrelated to type (e.g. proliferation, nuclear stains).

In [ ]:
RESOLUTION = 2.0
CLUSTER_MARKERS = None
cty.cluster(adata, resolution=RESOLUTION, markers=CLUSTER_MARKERS)

## 4. Cluster annotation → re-splitting mixed clusters

- Score = mean of the top k positive markers of the type − 0.7 × max negative marker (types·markers·k can be changed in `cty.MARKER_SETS`).
- `label`: top type. Score < 0.5 → `Unknown`; similar 1st·2nd with different lineages → `A|B?`.
- `cell_agree`: fraction of cells whose own top score matches the cluster type (low = the cluster may mix two types).
- `refine`: re-splits and re-annotates only clusters that are `A|B?` or have `cell_agree` < 0.6 (e.g. endothelial + plasma cells around vessels).

In [ ]:
tab = cty.annotate(adata)
tab = cty.refine(adata, tab)
tab

In [ ]:
fig = cty.plot_profiles(adata, tab); plt.show()

## 5. Manual fixes (optional)

Fix cluster types after looking at the table and heatmap above. e.g. `{'27.4': 'Glandular epi', '13': 'Unknown'}`.
Names not in `MARKER_SETS` get lineage 'Uncertain'.

In [ ]:
FIXES = {}
if FIXES:
    tab = cty.set_cluster_types(adata, tab, FIXES)
adata.obs['cell_type'].value_counts()

## 6. Checks

- If given labels exist, compare at lineage level (`GIVEN_TO_LINEAGE`: STELLAR label names → this module's lineages). Given labels were also assigned by people and are not ground truth — use this to find cells that disagree strongly.
- Cell map and type fractions.

In [ ]:
if 'cell_type_given' in adata.obs:
    cmp = cty.compare_labels(adata, 'cell_type_given', key='lineage', given_map=cty.GIVEN_TO_LINEAGE)
    fig = cty.plot_confusion(cmp); plt.show()

In [ ]:
fig = cty.plot_types(adata, 'cell_type'); plt.show()
fig = cty.plot_types(adata, 'cell_type', types=['B', 'Tfh', 'FDC', 'Plasma']); plt.show()   # highlight a few only
fig = cty.plot_composition(adata, 'cell_type'); plt.show()

## 7. Saving

In [ ]:
os.makedirs(f'{OUT}/codex', exist_ok=True)
adata.write_h5ad(f'{OUT}/codex/codex_types.h5ad')
tab.to_csv(f'{OUT}/codex/cluster_annotation.csv')
cols = ['cluster', 'cell_type', 'lineage', 'type_margin', 'type_agree', 'spill']
out = adata.obs[cols].copy()
out['x'], out['y'] = adata.obsm['spatial'][:, 0], adata.obsm['spatial'][:, 1]
out.to_csv(f'{OUT}/codex/cell_types.csv.gz')
print('saved:', os.listdir(f'{OUT}/codex'))

## 8. scRNA-seq preprocessing (RNA only)

Extracts the 6 GSE165860 tonsil scRNA samples (per-sample archives uploaded to `inputs`; a single GEO RAW tar also works) one by one: Cell Ranger raw matrix → AnnData for mapping.
Antibody (CITE-seq) features included in the matrix are dropped on reading.

| Section | Content | Defaults |
|---|---|---|
| 8-1 | extracting samples · reading · cell calling (UMI rank curve inflection) | `CELL_METHOD='inflection'` |
| 8-2 | QC: per-sample MAD (low UMI·genes, high MT%) + floors | 3 MAD, UMI ≥ 500, genes ≥ 250, MT ≤ 25 % |
| 8-3 | doublets: simulated-doublet kNN score, the expected 10x count from the top | 0.8 % per 1,000 cells |
| 8-4 | normalisation · HVG · Harmony integration | CP10k log1p, 3,000 HVGs, PCA 30 |
| 8-5 | clusters · types (same names as CODEX) · re-splitting if needed | Leiden 1.0 |
| 8-6 | saving | `outputs/rna/rna_prepared.h5ad` |

### 8-1 Extracting samples · cell calling

In [ ]:
import glob
# scRNA input: folder with per-sample archives (6 files such as GSM…_Tonsil_1a_scRNA_seq_raw_feature_bc_matrix_mtx.tsv.tar.gz).
# CODEX csv files etc. in the same folder lack 'scRNA' in the name and are ignored. If you uploaded a single GEO RAW tar, put that file path here
RNA_INPUT = f'{WORK_DIR}/inputs'
RNA_DIR = f'{WORK_DIR}/resources/scrna_mtx'             # extracted matrices (each sample extracted only once)
CELL_METHOD = 'inflection'                              # 'knee' is more conservative
DONOR = lambda s: s[:-1]                                # 'Tonsil_1a' → 'Tonsil_1' (replicates of the same donor)

found = sorted(glob.glob(f'{RNA_INPUT}/*scRNA*')) if os.path.isdir(RNA_INPUT) else [RNA_INPUT]
print(f'{len(found)} scRNA inputs:', *[os.path.basename(f) for f in found], sep='\n  ')
samples = rp.prepare_samples(RNA_INPUT, RNA_DIR, pattern='scRNA')   # extract per sample → {name: matrix folder}
rna, curves = rp.load_samples(samples, method=CELL_METHOD, batch_map={s: DONOR(s) for s in samples})  # per-sample reading·cell calling → merged
fig = rp.plot_barcode_ranks(curves); plt.show()

### 8-2 QC

Cells are not removed, only flagged with `qc_pass` / `qc_fail` (reason). Red dots = failing cells.

In [ ]:
qc_tab = rp.qc(rna, nmads=3, min_umi=500, min_genes=250, max_mt_abs=25)
display(qc_tab)
fig = rp.plot_qc(rna); plt.show()

### 8-3 doublet

- Score: how much cells mix in kNN with simulated doublets made by adding two real cells (Scrublet approach, per sample).
- Number called: the expected 10x doublet count from the top scores (`mode='expected'`). With `mode='otsu'`, in B-dominated tonsil, B+B simulated doublets could not be told from singlets and 15–25 % were over-called.
- Doublets of the same type are poorly caught in principle. The rest is checked again via `high_doublet` / `A|B?` clusters in 8-5.

In [ ]:
dbl_tab = rp.doublets(rna, mode='expected', rate_per_1000=0.008)
display(dbl_tab)
fig = rp.plot_doublets(rna); plt.show()

### 8-4 Normalisation · HVG · integration

- `layers['counts']` = original UMI, `layers['lognorm']` = log1p(CP10k). Mapping uses lognorm.
- HVG: per-sample variance z by mean bin → 3,000 genes commonly high across samples (MT·ribosomal·haemoglobin excluded).
- Harmony corrects only the embedding (`obsm['X_integrated']`). Expression values stay as is, so sample differences may remain in the mapped RNA.
- `mixing_score`: fraction of neighbours from other samples ÷ random expectation (1 = fully mixed).

In [ ]:
rna = rp.filter_cells(rna, drop_doublets=True)
rp.normalize(rna)
hvg_tab = rp.hvg(rna, n_top=3000, batch_key='sample')
rp.integrate(rna, batch_key='sample', n_pcs=30, method='harmony')
for key in ('sample', 'donor'):
    print(f"mixing ({key}): PCA {rp.mixing_score(rna, 'X_pca', key):.2f} → Harmony {rp.mixing_score(rna, 'X_integrated', key):.2f}")

### 8-5 Clusters · types

- Type names use the same vocabulary as CODEX (`rp.RNA_MARKERS`). Score = mean of the top half of between-cluster z, except
  - markers expressed in < 20 % of cells are −1 (so types almost absent from the sample are not picked by noise)
  - `RNA_NEGATIVE`: markers that must be absent (e.g. NK ← CD3), `RNA_REQUIRED`: markers that must be present (e.g. Treg ← FOXP3)
- `Unknown` / `A|B?` clusters are not used for mapping.
- If a rare type is buried in a large cluster, re-split only that cluster with `rp.refine` (in tonsil, Treg within the CD4 T cluster).

In [ ]:
RNA_RESOLUTION = 1.0
rp.cluster(rna, resolution=RNA_RESOLUTION)
rna_tab = rp.annotate(rna)
rp.embed_2d(rna)
display(rna_tab)
fig = rp.plot_marker_dots(rna, 'cluster'); plt.show()

In [ ]:
# re-split clusters hiding rare types after looking at the marker dot plot. e.g. a CD4 T cluster mixed with Treg (FOXP3)
REFINE = []                       # e.g. ['6', '9']
if REFINE:
    rna_tab = rp.refine(rna, rna_tab, REFINE, resolution=1.0)
    display(rna_tab)

RNA_FIXES = {}                    # e.g. {'11': 'CD8 T', '14': 'Unknown'}
if RNA_FIXES:
    rna_tab = rp.set_cluster_types(rna, rna_tab, RNA_FIXES)
for c in ('codex_type', 'sample', 'doublet_score'):
    fig = rp.plot_embedding(rna, c); plt.show()

### 8-6 Saving

In [ ]:
os.makedirs(f'{OUT}/rna', exist_ok=True)
rna.uns.pop('doublet_sim', None)          # save without the large array used for plotting
rna.write_h5ad(f'{OUT}/rna/rna_prepared.h5ad')
rna_tab.to_csv(f'{OUT}/rna/rna_cluster_annotation.csv')
print(rna.obs['codex_type'].value_counts())

## 9. CODEX ← scRNA mapping

**Restart point**: section 0 cells → this cell → from 9-1.

- Per CODEX type, the k scRNA cells of the same type with the most similar linked features (protein ↔ gene) → weighted mean.
- Types absent from scRNA or with fewer than 30 cells, `Unknown` and `A|B?` are not mapped (unit = none in `plan`).
- **Limitation**: differences between cells in the mapped RNA reflect reality only as far as the panel proteins explain them. Interpret position-dependent RNA differences only for types that beat random in the 9-3 validation.

In [ ]:
import anndata as ad
adata = ad.read_h5ad(f'{OUT}/codex/codex_types.h5ad')
rna = ad.read_h5ad(f'{OUT}/rna/rna_prepared.h5ad')
links = rm.linked_features(adata.var_names, rna.var_names)
print(f"{links.used.sum()} / {len(links)} linked features | markers without a corresponding gene in scRNA: {list(links.index[~links.used])}")
plan = rm.mapping_plan(adata, rna, allow_lineage=False)
plan

### 9-1 Mapping

- `METHOD='knn'`: kNN within type in a space where linked features are rank-normalised separately in each dataset (no installation, seconds).
- `METHOD='scmodal'`: kNN within type in the scMODAL shared embedding (scRNA uses linked features + HVGs). Run the installation cell below first; a GPU runtime is recommended (10,000 steps take long on CPU).

In [ ]:
# (only for scmodal) clone the repository and add it to the import path — instead of a conda environment. Needs annoy
SCMODAL_DIR = f'{WORK_DIR}/resources/scMODAL'
if not os.path.exists(SCMODAL_DIR):
    !git clone -q https://github.com/gefeiwang/scMODAL.git "{SCMODAL_DIR}"
!pip -q install annoy
sys.path.insert(0, SCMODAL_DIR)

In [ ]:
METHOD = 'knn'
K = 20
cmap = rm.map_cells(adata, rna, plan, links, k=K, method=METHOD, scmodal_dir=f'{WORK_DIR}/resources/scmodal_models')
cmap.stats

### 9-2 Validation: predicting held-out markers

Linked markers are split into 4 groups, each left out in turn and mapped → correlation (within type) between the mapped RNA of held-out markers and the actual protein.
`gain` = correlation − correlation with matches shuffled within the same type. gain ≈ 0 means no information beyond the 'type mean' is carried over.

In [ ]:
hold = rm.holdout_check(adata, rna, plan, links, k=K, method=METHOD)
display(hold.groupby('codex_type')[['r', 'null', 'gain']].median().round(3))
fig = rm.plot_holdout(hold); plt.show()

### 9-3 Saving · examples

In [ ]:
cmap.save(f'{OUT}/rna_map')
hold.to_csv(f'{OUT}/rna_map/holdout_check.csv', index=False)

GENES = ['CXCL13', 'MS4A1', 'CD3E']
imp = cmap.impute(rna, GENES)
xy = adata.obsm['spatial']
cmap_ = plt.get_cmap('viridis').copy(); cmap_.set_bad('lightgrey')
for g in imp.columns:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(xy[:, 0], xy[:, 1], c=imp[g].to_numpy(), s=0.3, cmap=cmap_, plotnonfinite=True, rasterized=True)
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_title(f'mapped {g} (lognorm; grey = not mapped)'); plt.show()

## 10. MaxFuse: label transfer → scRNA cell positions

Finds scRNA cell ↔ CODEX cell matches with MaxFuse (Chen et al. 2023). Unlike section 9 (kNN within type), it matches **from the two datasets alone without CODEX types**.

1. Matching: shared features (protein ↔ gene) + each dataset's cell graph → initial matches → refine with CCA → drop bad matches → propagate to the rest
2. **Label transfer**: per CODEX cell, distance-weighted vote of matched scRNA cell labels → `obs['mf_label']`. Validated with an agreement table against CODEX marker types
3. **Attaching positions**: per scRNA cell, positions of matched CODEX cells → `rna.obs['loc_x', 'loc_y']` (weighted centre), position of the closest match, spread, number of matches

Notes
- CODEX cells of types absent from scRNA (granulocytes·stroma·epithelium in this tonsil scRNA) still get forced matches → filter with the agreement table and `consistent`.
- One scRNA cell may match several CODEX cells (170 k CODEX vs 37 k scRNA). A large `spread` means the position is not a single place.
- Time: slow on CPU (local test: tens of minutes for 30 k CODEX cells). Try a subset with `MF_SUBSAMPLE` first and run everything when there is time.

In [ ]:
import anndata as ad
adata = ad.read_h5ad(f'{OUT}/codex/codex_types.h5ad')        # restart point
rna = ad.read_h5ad(f'{OUT}/rna/rna_prepared.h5ad')
links = rm.linked_features(adata.var_names, rna.var_names)

MF_SUBSAMPLE = 30000          # None = all CODEX cells
MF_SHARED_TYPES_ONLY = False  # True: match only CODEX cells of types present in scRNA (faster with fewer odd matches, but labels cannot be validated)

mask = np.ones(adata.n_obs, bool)
if MF_SHARED_TYPES_ONLY:
    mask &= adata.obs['cell_type'].astype(str).isin(set(rna.obs['codex_type'].astype(str))).to_numpy()
if MF_SUBSAMPLE and mask.sum() > MF_SUBSAMPLE:
    keep = np.random.default_rng(0).choice(np.where(mask)[0], MF_SUBSAMPLE, replace=False)
    mask = np.zeros(adata.n_obs, bool); mask[keep] = True
mf_res = mfm.run_maxfuse(adata, rna, links, codex_mask=mask, rna_key='codex_type')
mf_res.save(f'{OUT}/maxfuse')

### 10-1 Label transfer

- `agree`: among matched CODEX cells, fraction with transferred scRNA label = CODEX marker type.
- Confusion table rows = CODEX types. Rows of types absent from scRNA pile into odd columns such as B (expected — there is nothing to match).

In [ ]:
lt = mf_res.transfer_labels(adata, rna, rna_key='codex_type', codex_key='cell_type')
display(lt['confusion'])
fig = mfm.plot_label_confusion(lt); plt.show()
fig = cty.plot_types(adata[adata.obs['mf_label'].astype(str) != 'unmatched'], 'mf_label'); plt.show()

### 10-2 scRNA cell positions

- `x, y`: distance-weighted centre of matched CODEX cells, `best_x, best_y`: position of the closest match, `spread`: how spread out the matches are (coordinate units), `n_matches`.
- `consistent_only=True`: positions use only matches whose two types agree (so mismatched matches do not drag the position).

In [ ]:
loc = mf_res.rna_locations(adata, rna, consistent_only=False)
loc_c = mf_res.rna_locations(adata, rna, consistent_only=True)   # rna.obs['loc_*'] is overwritten with this result
display(loc_c.dropna().head())
fig = mfm.plot_rna_locations(adata, loc_c, rna, color='codex_type', use='best'); plt.show()

### 10-3 Saving · attaching expression to CODEX positions

In [ ]:
os.makedirs(f'{OUT}/maxfuse', exist_ok=True)
loc.to_csv(f'{OUT}/maxfuse/rna_locations_all_pairs.csv.gz')
loc_c.to_csv(f'{OUT}/maxfuse/rna_locations_consistent.csv.gz')
mf_res.pairs(adata, rna).to_csv(f'{OUT}/maxfuse/pairs.csv.gz', index=False)
adata.obs[['mf_label', 'mf_conf']].to_csv(f'{OUT}/maxfuse/codex_transferred_labels.csv.gz')
rna.write_h5ad(f'{OUT}/maxfuse/rna_with_location.h5ad')

mf_map = mf_res.to_cellmap(adata, rna, consistent_only=True)     # used like the section 9 CellMap
imp = mf_map.impute(rna, ['CXCL13', 'MS4A1', 'CD3E'])
print(imp.notna().all(1).sum(), 'CODEX cells with expression')

## 11. Niche remapping: type mean (evidence) → within-type remapping by adjacent-type signal hypotheses (hypothesis) → judged with CODEX

1. **Type-mean mapping**: per CODEX cell, the mean of scRNA cells of the same type. No within-type differences (evidence result).
2. **Adjacent type pairs**: type composition of 10 spatial neighbours → (receiver ← sender) pairs significantly more adjacent than a label-shuffled expectation.
3. **NicheNet**: ligands expressed by the sender type that best explain substate differences (genes of interest) in the receiver type's scRNA → per-cell score of their downstream target genes.
4. **Remapping**: adjusts candidate weights so scRNA cells with high target scores land where receiver-type CODEX cells have many sender-type neighbours (protein candidates × hypothesis kernel × balanced cell use). Whole cell profiles are moved, not per-gene orderings.
5. **Judgement**: remap with signal-response proteins (PD-L1, IDO, HLA-DR, COX2, CD25, PD-1, CD38, BCL2) removed from the protein features → does the spatial pattern of those genes match the actual protein better, compared with a null shuffling the CODEX-side hypothesis.
6. **Routes**: NicheNet passes → `nichenet` / not possible·fails → try similarity (closeness to the sender type in Harmony space) → `similarity` (`similarity_weak` after a NicheNet judgement failure) / all fail → `none`. If the sender type is absent from scRNA, neither route can be computed (`sender_absent`).

**Interpretation**: `remap` is a hypothesis map. Only pairs passing in `routes` have data support; other within-type differences are at the `baseline` (protein only) level.

In [ ]:
import anndata as ad
adata = ad.read_h5ad(f'{OUT}/codex/codex_types.h5ad')        # restart point
rna = ad.read_h5ad(f'{OUT}/rna/rna_prepared.h5ad')
links = rm.linked_features(adata.var_names, rna.var_names)
plan = rm.mapping_plan(adata, rna)

USE_NICHENET = True                                            # False: similarity route only
prior = nr.load_nichenet(f'{WORK_DIR}/resources/nichenet') if USE_NICHENET else None   # downloads about 260 MB the first time

### 11-1 Type-mean mapping (evidence)

In [ ]:
GENES = ['CXCL13', 'CD274', 'IDO1', 'MS4A1', 'CD3E']
mean_imp = nr.mean_impute(adata, rna, plan, GENES)
print(f"CODEX cells with mean mapping {mean_imp.notna().all(1).sum():,} / {adata.n_obs:,}")

### 11-2 Remapping · judgement

Time: 7 receiver types × at most 6 sender types × (baseline + hypothesis + `N_NULL` nulls). Tens of minutes on local CPU with `N_NULL=20`.
Look at the route table with `N_NULL=5` first, then use 20 for the final run (smallest p = 1/(N_NULL+1)).

In [ ]:
N_NULL = 20
res = nr.run_niche_remap(adata, rna, plan, links, prior, n_null=N_NULL, sigma=0.15)
res.save(f'{OUT}/niche')
display(res.routes[['receiver', 'sender', 'adj_ratio', 'route', 'ligands', 'n_targets', 'gain', 'p', 'best_protein', 'reason']])

In [ ]:
fig = nr.plot_adjacency(res.adjacency, types=sorted(plan.index[plan.unit == 'type']) +
                        [t for t in ['Granulocyte', 'DC', 'FDC', 'PDPN stroma', 'Fibroblast', 'Endothelial'] if t in plan.index]); plt.show()
fig = nr.plot_routes(res.routes); plt.show()
v = res.validation
display(v.pivot_table(index=['receiver', 'sender', 'route'], columns='protein', values='gain').round(3))

### 11-3 Ligand activity · viewing remapping results

- `res.activities[type]`: ligand ranking of the receiver type (only `aupr_corrected` > 0 is meaningful).
- `res.remap.impute(rna, genes)` − `res.baseline.impute(...)`: the part changed by the hypothesis. Check with ligand target genes of passing pairs.

In [ ]:
for r, a in res.activities.items():
    print(r, ', '.join(f'{l} {x:.3f}' for l, x in a.aupr_corrected.head(8).items()))

ok = res.routes[res.routes.route != 'none']
if len(ok):
    x = ok.iloc[0]
    gene = nr.VALIDATION_PROTEINS and rm.PROTEIN_TO_GENE.get(x.best_protein, [None])[0]
    if gene:
        d = res.remap.impute(rna, [gene])[gene] - res.baseline.impute(rna, [gene])[gene]
        fig = nr.plot_remap(adata, d, title=f'{gene}: remap − baseline ({x.receiver} ← {x.sender}, {x.route})'); plt.show()

## 12. Benchmark: held-out marker test · niche remapping judgement

**Restart point**: section 0 cells → this section. Reads the saved results (`outputs/niche`) instead of recomputing sections 9–11 (the held-out marker test redoes mapping per fold).

| Test | Headline row | Baseline | Verdict |
|---|---|---|---|
| **Held-out markers** (method comparison) | `median marker gain (method)`: Spearman r between mapped RNA of held-out markers ↔ actual protein minus the null, median across types per marker → median across markers | type-mean mapping (mapping rows shuffled within the same type, gain = 0) | marker gain > 0 (one-sided Wilcoxon, per marker) p ≤ 0.05 |
| | `gain method − ref`: paired per-marker differences | `BENCH_METHODS[0]` | paired one-sided Wilcoxon p ≤ 0.05 |
| **Niche remapping** | `passed tests`: number of passing (receiver, sender, route) | chance expectation = tests × α | binomial test p ≤ 0.05 |
| | `mean excess over random placement`: (gain − null mean) over **all** attempted tests, averaged over judging proteins | remapping with the hypothesis shuffled among receiver cells (= random placement) | greater than 0 (one-sided Wilcoxon, per test) |
| | `tests passing BH q ≤ 0.1`, baseline-map ρ | – | reference |

* **Fairness**: every method is mapped with the same folds (held-out marker groups), and only **cells mapped by every method** in that fold are compared. scMODAL·MaxFuse are retrained per fold since held-out markers must not enter training (slow: MaxFuse about 3–4 min per fold with a 4,000-cell CODEX subset, local CPU).
* Cells where one protein value covers ≥ 90 % within a type (e.g. CK7 in macrophages — almost all negative) carry no information and are dropped.
* **Read both niche rows together.** The pass count alone can look good because it picks out a few pairs. `mean excess` ≤ 0 means 'hypothesis-based remapping is not better than random placement overall; only some pairs passed'.
* If `N_NULL` in section 11 is small, the smallest p (1/(N_NULL+1)) is large and BH cannot be passed. The note states the p needed.
* Accumulate runs with different settings (method, `K` etc.) under different `BENCH_RUN` names and the last table becomes a run × metric comparison. Output: `outputs/benchmark/benchmark.csv`, detail tables in `outputs/benchmark/<run>/`.


In [ ]:
import anndata as ad, map_benchmark as mb
importlib.reload(mb)
if 'adata' not in globals() or 'rna' not in globals():           # restart point
    adata = ad.read_h5ad(f'{OUT}/codex/codex_types.h5ad')
    rna = ad.read_h5ad(f'{OUT}/rna/rna_prepared.h5ad')
links = rm.linked_features(adata.var_names, rna.var_names)
plan = rm.mapping_plan(adata, rna, allow_lineage=False)              # same plan as section 9

BENCH_RUN     = 'knn k=20'          # name of this setting (change it when settings change)
BENCH_METHODS = ['knn']             # first = reference. 'scmodal', 'maxfuse' can be added (retrained per fold → slow)
BENCH_K       = 20
MF_BENCH_SUBSAMPLE = 4000           # number of CODEX cells MaxFuse matches when included (shared cells shrink to this too)

mf_mask = None
if 'maxfuse' in BENCH_METHODS and MF_BENCH_SUBSAMPLE:
    mf_mask = np.zeros(adata.n_obs, bool)
    mf_mask[np.random.default_rng(0).choice(adata.n_obs, min(MF_BENCH_SUBSAMPLE, adata.n_obs), replace=False)] = True

bm = mb.Benchmark(BENCH_RUN)
HC = mb.holdout_compare(adata, rna, plan, links, methods=BENCH_METHODS, k=BENCH_K, n_folds=4, n_null=5,
                        maxfuse_mask=mf_mask, scmodal_dir=f'{WORK_DIR}/resources/scmodal_models')
HC.to_benchmark(bm)
HC.save(f"{OUT}/benchmark/{BENCH_RUN.replace(' ', '_').replace('/', '_')}")
RJ = mb.remap_judgement(bm, res if 'res' in globals() else f'{OUT}/niche')

pd.set_option('display.max_colwidth', 90)
display(bm.headline())
display(RJ['tests'].sort_values('p').head(20) if RJ else None)
bm.save(f'{OUT}/benchmark/benchmark.csv')
print('setting × headline metrics (all runs saved so far):')
display(mb.load_runs(f'{OUT}/benchmark/benchmark.csv'))


## License

The code of this pipeline (this notebook and its modules) is released under the MIT License below, also in `LICENSE`.
Third-party packages and datasets it installs or downloads keep their own licenses — see `THIRD_PARTY_NOTICES.md`.

```text
MIT License

Copyright (c) 2026 <AUTHOR NAME>

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```
